# Fairness Preprocessing Tutorial (Regression)

This tutorial demonstrates how to apply the **fairness preprocessing** method in PipeGenie for a **regression** task.

Unlike classification, the target variable is continuous. Before applying the preprocessing method, the sensitive feature is prepared using the helper function `prepare_data()`, which creates fairness groups suitable for regression.

The model is then evaluated using both predictive performance and fairness metrics.

## 1. Load and Prepare the Dataset

We begin by loading the **Communities and Crime** dataset and defining:

- **X**: input features
- **y**: continuous target variable
- **A**: sensitive feature

For this tutorial, the helper function `prepare_data()` is used to prepare the regression dataset and generate the sensitive groups required for fairness preprocessing.

Then we split the dataset into training and testing sets.

In [1]:
import pandas as pd
from pipegenie.fairness import prepare_data

df = pd.read_csv("../datasets/communities.data", header=None)
    
df = df.rename(columns={7: "racepctblack", 127: "ViolentCrimesPerPop"})

y = df["ViolentCrimesPerPop"]

A = prepare_data(df["racepctblack"],n_groups=4,labels=["Q1", "Q2", "Q3", "Q4"])

X = df.drop(columns=["ViolentCrimesPerPop","racepctblack"])
X = pd.get_dummies(X, drop_first=True)

In [2]:
from pipegenie.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)
A_train, A_test, _, _ = train_test_split(A,y,test_size=0.25,random_state=42)

## 2. Apply Fairness Preprocessing

The fairness preprocessing step is applied **before training**.

Its objective is to reduce bias across the sensitive groups while preserving as much predictive performance as possible.

In [3]:
from pipegenie.fairness.preprocessing import fairness_preprocessing

X_train_fair, y_train_fair = fairness_preprocessing(X_train,y_train,A_train)

## 3. Create the PipeGenie Regressor

Next, we initialize a `PipegenieRegressor` using the regression grammar.

The grammar defines the search space explored by PipeGenie to automatically generate regression pipelines.

In [4]:
grammar_file = "../../grammars/sample-grammar-regression.xml"

from pipegenie.regression import PipegenieRegressor

model = PipegenieRegressor(
grammar=grammar_file,
generations=5,
pop_size=50,
elite_size=5,
maximization=False,
n_jobs=5,
seed=9,
outdir="samples/regression_1",
)

## 4. Train the Model

The regression model is trained using the fairness-preprocessed training data.

In [5]:
model.fit(X_train_fair, y_train_fair)

                                         fitness                                             size                                        fitness_elite                                      size_elite              
                 -------------------------------------------------------    --------------------------------------    ----------------------------------------------------    --------------------------------------
gen    nevals    min           max            avg            std            min    max    avg           std           min           max           avg           std           min    max    avg           std       
0      50        0.080689      1.1815e+123    3.2821e+121    1.9417e+122    1      6      2.88          1.3363        0.080689      0.098059      0.094213      0.0067922     2      4      3             0.63246   
1      42        0.080689      0.27053        0.12297        0.043001       1      6      2.74          1.1629        0.080689      0.097978      0.

## 5. Generate Predictions

After training, predictions are generated on the test set using the trained PipeGenie regressor.

In [6]:
y_pred = model.predict(X_test)

## 6. Generate the Fairness Report

The fairness report summarizes:

- Overall model performance
- Performance for each sensitive group

This allows us to evaluate whether the preprocessing method reduced disparities between groups.

In [7]:
from pipegenie.fairness import fairness_report
from pipegenie.metrics import root_mean_squared_error

# =====================================================
# FAIRNESS REPORT
# =====================================================

report = fairness_report(
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=A_test,
    metric=root_mean_squared_error
)

print("\n==============================")
print("OVERALL PERFORMANCE")
print("==============================")

print(report["overall"])

print("\n==============================")
print("PERFORMANCE BY GROUP")
print("==============================")

print(report["by_group"])


OVERALL PERFORMANCE
0.1390691933276418

PERFORMANCE BY GROUP
sensitive_feature_0
Q1    0.089693
Q2    0.110240
Q3    0.132705
Q4    0.199033
Name: root_mean_squared_error, dtype: float64


## 7. Evaluate Model Performance

Finally, we evaluate the regression model using two complementary performance metrics:

- **Model Score (R²):** Measures how well the model explains the variance in the target variable. A value closer to **1** indicates a better fit.
- **Root Mean Squared Error (RMSE):** Measures the average prediction error. Lower RMSE values indicate better predictive performance.

These metrics provide an overall assessment of the model before interpreting the fairness results.

In [8]:
# =====================================================
# STANDARD METRICS
# =====================================================

print("\n==============================")
print("ROOT MEAN SQUARED ERROR")
print("==============================")

print(
    root_mean_squared_error(
        y_test,
        y_pred
    )
)

print("\n==============================")
print("MODEL SCORE (R²)")
print("==============================")

print(
    model.score(
        X_test,
        y_test
    )
)


ROOT MEAN SQUARED ERROR
0.1390691933276418

MODEL SCORE (R²)
0.6365334986248519


## Conclusion

In this tutorial, we:

- Prepared a regression dataset using the helper function `prepare_data()`.
- Applied fairness preprocessing.
- Trained a PipeGenie regressor.
- Generated predictions.
- Evaluated predictive performance.
- Assessed fairness across the sensitive groups.

This workflow demonstrates how fairness preprocessing can be integrated into a standard regression pipeline in PipeGenie.